# 🎬 Workshopstart — Lea, signalet og baseline

**Lea er i ferd med å forsvinne.**

Hun er en av StreamNords mest aktive brukere. Hun elsker nordisk indie-drama
og mørkere katalogtitler — men får de samme blockbusterne som alle andre.
Produktteamet trenger et svar til møtet i morgen: **hva bør vi bygge?**

Du er den nye ML-ingeniøren hos StreamNord — en norsk strømmetjeneste
med ~5 000 aktive brukere og ~10 000 filmer i katalogen.
Denne notatboken gjør fire ting: etablerer caset, forklarer signalet,
setter opp evalueringen og måler den sterkeste enkle baselinen.

---

### Konsolidert workshopløp

| Notebook | Rolle i workshopen |
|---|---|
| `00_velkommen` | Case, signal, metrikker og baseline |
| `01_content_based` | Første personaliserte modell |
| `02_collaborative_filtering` | CF og matrix factorization |
| `03_hybrid_systems` | Hybrider, kontekst og fairness |
| `04_ship_decision` | Sluttanbefaling og produksjonsvalg |

Tyngre eller mer spesialisert stoff ligger som **valgfri appendix** inne i de relevante notebookene.

## 1. Verifiser miljøet

In [ ]:
import sys
print(f'Python {sys.version}')

import pandas as pd
import numpy as np
import scipy
import sklearn
import matplotlib

print(f'pandas {pd.__version__}')
print(f'numpy {np.__version__}')
print(f'scipy {scipy.__version__}')
print(f'scikit-learn {sklearn.__version__}')
print(f'matplotlib {matplotlib.__version__}')

try:
    import implicit
    print(f'implicit {implicit.__version__}')
except ImportError:
    print('implicit not installed')

try:
    import faiss
    print('faiss OK')
except ImportError:
    print('faiss-cpu not installed')

print('\nAlle importer OK')

## 2. Last ned og klargjør data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from data.sample_ml25m import sample_and_save, OUT_DIR

if (OUT_DIR / 'interactions.parquet').exists():
    print(f'Data finnes allerede i {OUT_DIR}')
else:
    sample_and_save()

## 3. Last inn interaksjoner og metadata

In [ ]:
from src.data import load_interactions, load_item_metadata, GENRE_COLS

interactions = load_interactions()
items = load_item_metadata()

n_users = interactions.user_id.nunique()
n_items = interactions.item_id.nunique()
n_interactions = len(interactions)
sparsity = 1 - n_interactions / (n_users * n_items)

print(f'Interaksjoner: {n_interactions:,} rader')
print(f'Brukere:       {n_users:,}')
print(f'Filmer:        {n_items:,}')
print(f'Sparsitet:     {sparsity:.2%}')

interactions.head()

### Hva betyr sparsiteten?

Over 99 % av matrisen er tom. Det betyr at en gjennomsnittlig bruker har sett
kanskje 50 av 10 000 filmer. Det meste vi *ikke* ser er ukjent — ikke mislikt.
Denne forskjellen er viktig: modellene våre må lære av det lille vi har,
uten å anta at fravær av klikk betyr avvisning.

## 4. 📊 Møt Lea og Jonas

Lea (bruker 451) er en av StreamNords mest trofaste brukere — men de siste
ukene har hun nesten sluttet å klikke. La oss se hva hun faktisk har sett.

Jonas (bruker 102) elsker blockbustere — alt fra Marvel til Star Wars.
Han klikker på det meste som ligger på forsiden. Disse to representerer
ytterpunktene av brukerbasens vår: **nisjesmak vs mainstream**.

In [ ]:
# Disse ID-ene er spesifikke for vårt utvalgsdatasett (ml-25m-sample).
# Asserts under fanger det opp hvis datasettet regenereres og de ikke lenger finnes.
LEA_ID = 451
JONAS_ID = 102

assert LEA_ID in interactions.user_id.values, f'LEA_ID={LEA_ID} finnes ikke i datasettet'
assert JONAS_ID in interactions.user_id.values, f'JONAS_ID={JONAS_ID} finnes ikke i datasettet'

lea_items = interactions[interactions.user_id == LEA_ID].merge(items, on='item_id')
jonas_items = interactions[interactions.user_id == JONAS_ID].merge(items, on='item_id')
print(f'Lea har sett {len(lea_items)} filmer. Jonas har sett {len(jonas_items)} filmer.')
print(f'\nLeas siste 10 filmer:')
lea_items.tail(10)[['title'] + GENRE_COLS[:6]]

In [ ]:
print('Jonas sine siste 10 filmer:')
jonas_items.tail(10)[['title'] + GENRE_COLS[:6]]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lea_genre_dist = lea_items[GENRE_COLS].sum().sort_values(ascending=True)
lea_genre_dist.plot.barh(ax=axes[0], color='coral')
axes[0].set_xlabel('Antall filmer')
axes[0].set_title('Lea — sjangerfordeling')

jonas_genre_dist = jonas_items[GENRE_COLS].sum().sort_values(ascending=True)
jonas_genre_dist.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Antall filmer')
axes[1].set_title('Jonas — sjangerfordeling')

plt.tight_layout()
plt.show()

## 🏋️ Før vi kjører noen modell

Skriv ned gjetningene dine **før** du ser resultatene.
Vi kommer tilbake til disse i notebook 04 for å se om du hadde rett.

Vær så spesifikk du kan — det er lærerikt å ta feil.

In [ ]:
# DINE GJETNINGER — fyll inn før du kjører neste celle
#
# 1. Hva slags filmer tror du Lea vil ha anbefalt?
#    Svar: 
#
# 2. Hva tror du popularitetslisten vil inneholde?
#    Svar: 
#
# 3. Vil popularitet fungere bedre for Lea eller Jonas? Hvorfor?
#    Svar: 
#
# 4. Hvor sikker er du på gjetning 3? (1 = ren gjetning, 5 = helt sikker)
#    Sikkerhet: 
#
# Tips: Kom tilbake hit i notebook 04 og sammenlign.

## 5. Hvilket signal har vi?

### Eksplisitt vs implisitt feedback

| Type | Eksempler | Fordel | Problem |
|---|---|---|---|
| **Eksplisitt** | Stjerner, likes, tommel opp | Tydelig preferanse | Sjelden i virkelige produkter |
| **Implisitt** | Klikk, visning, kjøp, avspilling | Finnes i store mengder | Manglende data er tvetydig |

StreamNord-dataene våre er hovedsakelig **implisitte**. Tenk på det slik:
en rating er en melding — et klikk er bare et blikk. Vi har bare blikkene.

Vi vet at Lea så noe, men ikke sikkert om hun aktivt likte det.
Derfor betyr manglende interaksjon ikke automatisk avvisning.

## 6. Evalueringsoppsett

Vi løser et **topp-N-rangeringsproblem**: gitt en bruker, ranger alle filmer
slik at det brukeren *faktisk* vil se havner blant topp K.

### Leave-one-out-splitt

Vi gjemmer den siste filmen hver bruker så, og spør: ville modellen funnet den?
Vi bruker *siste*, ikke tilfeldig, fordi vi vil teste om modellen kan forutsi
hva som skjer *neste* — akkurat som i et ekte produkt.

### Metrikker

- **Recall@K**: var den gjemte filmen blant topp-K? (traff vi i det hele tatt?)
- **NDCG@K**: lå treffet høyt i listen, eller måtte brukeren scrolle?
- **MAP@K**: belønner modeller som konsekvent plasserer treff tidlig

Vi bruker K=10 — en typisk forsidelengde i en strømmetjeneste.

In [ ]:
from src.split import leave_one_out_split, build_sparse_matrix
from src.metrics import recall_at_k, ndcg_at_k, map_at_k

train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10

print(f'Trening: {len(train_df):,} interaksjoner')
print(f'Test:    {len(test_df):,} interaksjoner (1 per bruker)')
print(f'Matrise: {train_matrix.shape}, nnz={train_matrix.nnz:,}')

## 7. Den late anbefalingen

"Gi alle det samme." Popularitet er den enkleste strategien —
og overraskende vanskelig å slå. La oss se hva den gjør med Lea.

In [ ]:
item_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
global_ranking = np.argsort(-item_counts)

def recommend_popular(train_matrix, user_ids, k=10):
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index, user_id in enumerate(user_ids):
        seen = set(train_matrix[user_id].indices)
        unseen_popular = [item_id for item_id in global_ranking if item_id not in seen][:k]
        recommendations[row_index] = unseen_popular
    return recommendations

recs_pop = recommend_popular(train_matrix, user_ids, k=K)
recall_value = recall_at_k(recs_pop, test_items, K)
ndcg_value = ndcg_at_k(recs_pop, test_items, K)
map_value = map_at_k(recs_pop, test_items, K)

print(f'Popularitet: Recall@{K}={recall_value:.4f}  NDCG@{K}={ndcg_value:.4f}  MAP@{K}={map_value:.4f}')

### Er det bra?

Tallet alene sier lite — det trenger en referanse. Recall@10 her betyr:
*for hvor stor andel av brukerne lå den gjemte filmen blant topp 10?*
Med ~10 000 filmer er ren gjetting nær null (se appendix), så popularitet
er allerede et stykke over bunnen. Det er **baselinen** alt vi bygger
senere må slå for å fortjene kompleksiteten sin.

> 💡 **Recsys-oppskriften:** popularitet dekker allerede de to første stegene —
> **kandidatgenerering** (alle usette filmer) og **rangering** (sortér på antall visninger).
> Det vi mangler er personalisering i rangeringen og **re-rangering** for mangfold.
> Den samme oppskriften — kandidatgenerering → rangering → re-rangering — går igjen
> i hver modell resten av workshopen.

In [ ]:
lea_idx = np.where(user_ids == LEA_ID)[0][0]
jonas_idx = np.where(user_ids == JONAS_ID)[0][0]

overlap_lea_jonas = len(set(recs_pop[lea_idx]) & set(recs_pop[jonas_idx])) / K
print(f'Overlapp Lea vs Jonas: {overlap_lea_jonas:.0%}')
print('  → Begge får nesten identiske anbefalinger.\n')

for name, uid, idx in [('Lea', LEA_ID, lea_idx), ('Jonas', JONAS_ID, jonas_idx)]:
    titles = items.set_index('item_id').loc[recs_pop[idx]]
    print(f'Popularitetsanbefalinger til {name} (bruker {uid}):')
    for rank, (_, row) in enumerate(titles.iterrows(), 1):
        genres = [g for g in GENRE_COLS if row.get(g, 0) == 1]
        print(f'  {rank:>2}. {row["title"]}  [{", ".join(genres[:3])}]')
    print()

> *Marte ser på listene:* «Vent — Lea og Jonas får *nesten det samme*?
> Hun ser gamle filmer, han ser blockbustere, men vi anbefaler identisk?
> **Noe burde gjøres.**»

In [ ]:
# Sjangermismatch-visualisering: hva Lea liker vs hva hun får anbefalt
rec_items_lea = items.set_index('item_id').loc[recs_pop[lea_idx]]
rec_genres_lea = rec_items_lea[GENRE_COLS].sum()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(GENRE_COLS))
width = 0.35
lea_profile = lea_items[GENRE_COLS].sum()
lea_norm = lea_profile / lea_profile.sum()
rec_norm = rec_genres_lea / rec_genres_lea.sum()
ax.barh(x - width/2, lea_norm, width, label='Leas faktiske smak', color='coral')
ax.barh(x + width/2, rec_norm, width, label='Popularitetsanbefalinger', color='gray')
ax.set_yticks(x)
ax.set_yticklabels(GENRE_COLS)
ax.set_xlabel('Andel')
ax.set_title('Sjangermismatch: hva Lea liker vs hva hun får')
ax.legend()
plt.tight_layout()
plt.show()

## Valgfri appendix — tilfeldig baseline

Denne delen kan brukes hvis dere vil vise en helt naiv nedre grense.
I kjerneflyten kan den hoppes over.

In [ ]:
def recommend_random(train_matrix, user_ids, k=10, seed=42):
    rng = np.random.default_rng(seed)
    all_items = np.arange(train_matrix.shape[1])
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index, user_id in enumerate(user_ids):
        unseen = np.setdiff1d(all_items, train_matrix[user_id].indices)
        recommendations[row_index] = rng.choice(unseen, size=k, replace=False)
    return recommendations

recs_rand = recommend_random(train_matrix, user_ids, k=K)
print(f"{'Modell':<14} {'Recall@10':>10} {'NDCG@10':>10} {'MAP@10':>10}")
print('-' * 48)
for name, recs in [('Popularitet', recs_pop), ('Tilfeldig', recs_rand)]:
    recall_value = recall_at_k(recs, test_items, K)
    ndcg_value = ndcg_at_k(recs, test_items, K)
    map_value = map_at_k(recs, test_items, K)
    print(f'{name:<14} {recall_value:>10.4f} {ndcg_value:>10.4f} {map_value:>10.4f}')

---

> *Marte:* «Bra. Nå vet vi hva signalet betyr, og vi ser at popularitet ikke er nok.
> Men Lea har en tydelig smak — sjangerprofilen viser det. Kan vi bruke den informasjonen
> direkte til å bygge noe bedre?»

*Vi kommer også tilbake til to ting som er like viktige som treffsikkerhet:
**cold start** (nye brukere og filmer) og **fairness** — begge i notebook 03–04.*

**Neste steg** → `01_content_based.ipynb`